# MergerRetriever and LongContextReorder

## What This Notebook Covers

This notebook builds a multi-source RAG (Retrieval Augmented Generation) pipeline that queries two completely separate knowledge bases simultaneously and applies intelligent post-processing to the combined results before passing them to an LLM.

The notebook is organised into six categories.

Category 1 handles all installations and environment setup.

Category 2 loads two separate PDF books and splits them into chunks ready for embedding.

Category 3 converts all chunks into dense vectors using two different embedding models and stores them in separate ChromaDB collections.

Category 4 creates individual retrievers for each book, merges them using MergerRetriever (LOTR), and demonstrates the raw merged output before any filtering.

Category 5 adds EmbeddingsRedundantFilter and LongContextReorder on top of the merged retriever to remove duplicate chunks and solve the lost-in-the-middle problem.

Category 6 loads TinyLlama, builds an LCEL RAG chain, runs queries against both books, and displays answers with source attribution.

## The Core Problem This Notebook Solves

A standard RAG system has one retriever backed by one vector store. If your knowledge spans multiple documents or databases you would need to query each one separately and manually combine results.

MergerRetriever solves this by acting as a single unified retriever that internally fans out to any number of individual retrievers and merges all their results automatically.

The merged output however contains up to 10 chunks (5 per retriever) that may overlap in content. The compression pipeline in Category 5 removes semantic duplicates and reorders survivors so the most relevant chunks appear at the start and end of the context window where LLMs pay the most attention.

## Category 1  Installation and Environment Setup

### Why These Libraries Are Needed

langchain is the core framework providing document loaders, text splitters, retrievers, chains, and the LCEL pipe syntax used throughout this notebook.

chromadb is a lightweight local vector database that stores document embeddings on disk and supports fast approximate nearest-neighbor search at query time.

huggingface_hub and sentence-transformers provide access to pre-trained embedding models that run locally without any API calls or costs.

pypdf is the underlying library that PyPDFLoader uses internally to parse PDF binary format and extract raw text page by page.

tiktoken is a tokenizer used by some LangChain chains internally when estimating token counts.

langchain-huggingface provides the updated HuggingFacePipeline class for wrapping local HuggingFace models as LangChain LLM objects.

python-dotenv loads API keys from a local .env file so secrets are never hardcoded in the notebook.

In [ ]:
# Install all required libraries
# Run this cell once. After installation, restart the kernel before running any other cell.
!pip install -qU langchain chromadb huggingface_hub sentence-transformers pypdf tiktoken
!pip install -U langchain-community
!pip install langchain-huggingface
!pip install langchain-text-splitters
!pip install python-dotenv

### Loading Environment Variables

API keys must never be hardcoded in a notebook. If the notebook is pushed to GitHub anyone who finds the repository can use your paid API quota. The python-dotenv library reads a local .env file that is listed in .gitignore and injects each key as an environment variable.

The .env file in the project folder should contain:
```
OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxx
HF_TOKEN=hf_xxxxxxxxxxxxxxxx
```

In [ ]:
from dotenv import load_dotenv
import os

# load_dotenv reads the .env file from the current working directory
# and injects every key-value pair as an OS environment variable
# override=True ensures fresh values from .env overwrite any stale shell variables from a previous session
load_dotenv(override=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

print('Working directory:', os.getcwd())

## Category 2  Data Loading and Text Splitting

### Loading Two Separate Knowledge Bases

This notebook uses two PDFs as separate knowledge bases: a Harry Potter book and a Game of Thrones book. The fundamental goal is to build one system that can answer questions accurately from either book without mixing up information or losing context.

PyPDFLoader reads a PDF file page by page and returns a Python list of LangChain Document objects. Each Document carries two attributes. page_content is a string containing the raw text extracted from that page. metadata is a dictionary automatically populated with the source file path and the page number. This metadata persists through the entire pipeline so at the end we can trace each retrieved chunk back to its exact page and source file.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Verify the current working directory so we know where to place the PDF files
print(os.getcwd())

In [ ]:
# Load the Harry Potter PDF from the Data subfolder
# Adjust this path to match where you have stored the file on your machine
loader_harrypotter = PyPDFLoader(
    r"F:\Advance RAG\Data\harry_potter_book.pdf"
)

In [ ]:
# load() reads every page of the PDF and returns a list of Document objects
# One Document per page, each with page_content and metadata fields
document_harrypotter = loader_harrypotter.load()

In [ ]:
# Check how many pages were successfully loaded from the Harry Potter PDF
print(len(document_harrypotter))

In [ ]:
# Load the Game of Thrones PDF
loader_got = PyPDFLoader(r"F:\Advance RAG\Data\got_book.pdf")

In [ ]:
document_got = loader_got.load()

In [ ]:
# Check how many pages were loaded from the GOT PDF
print(len(document_got))

### Why We Split Documents Into Chunks

LLMs and embedding models both have maximum token limits. A single PDF page can contain 2000 to 4000 characters which is too long for most embedding models to encode accurately in a single pass. Feeding an entire page as one unit also makes retrieval imprecise because the retrieved unit is too coarse.

RecursiveCharacterTextSplitter breaks every page into smaller overlapping pieces. It tries to split on paragraph boundaries first, then sentence boundaries, then word boundaries, working recursively until each piece is within the target size.

chunk_size=500 means each chunk contains at most 500 characters of text.

chunk_overlap=100 means the last 100 characters of each chunk are repeated at the start of the next chunk. This overlap ensures that a sentence or key phrase at a chunk boundary is still fully represented in at least one chunk and is not silently cut in half.

# Create the Text Splitter for Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure the splitter
# chunk_size: maximum characters per chunk
# chunk_overlap: characters shared between end of one chunk and start of the next
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
# Split all Harry Potter pages into chunks
# Output is still a list of Document objects but with shorter page_content
# Metadata from the parent page (source path, page number) is inherited by every child chunk
text_harrypotter = text_splitter.split_documents(document_harrypotter)

In [ ]:
# Split all GOT pages into chunks
text_got = text_splitter.split_documents(document_got)

In [ ]:
# Check how many chunks were produced from Harry Potter
print(len(text_harrypotter))

In [ ]:
# Check how many chunks were produced from GOT
print(len(text_got))

## Category 3  Embedding Models and ChromaDB Vector Stores

### Two Embedding Models With Different Roles

This notebook uses two different embedding models, each chosen for a specific role in the pipeline.

HuggingFaceEmbeddings with all-MiniLM-L6-v2 is a fast and lightweight model that produces 384-dimensional vectors. It is used inside the EmbeddingsRedundantFilter in Category 5 because that filter needs to compare many pairs of chunks quickly. Speed matters more than raw embedding quality in the filtering context.

HuggingFaceBgeEmbeddings with BAAI/bge-large-en is a more powerful model that produces 1024-dimensional vectors. BGE stands for BAAI General Embedding, developed by the Beijing Academy of Artificial Intelligence. It is specifically optimised for retrieval tasks and produces higher-quality embeddings than MiniLM. It is used for ingesting documents into ChromaDB and building the retrievers because retrieval quality has the biggest impact on final answer accuracy.

Using a stronger embedding model for the vector store and a faster one for filtering is a deliberate performance-quality trade-off that is common in production RAG systems.

# Load the Embedding Model to Convert the Data into Vectors

In [ ]:
from langchain_community.embeddings import (
    HuggingFaceEmbeddings,
    HuggingFaceBgeEmbeddings,
)

In [ ]:
# Initialize the Hugging Face embedding model.
# Uses the all-MiniLM-L6-v2 model to convert text into dense vector embeddings
# for semantic search, retrieval, and RAG applications.
minilm_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
# Initialize the BGE (BAAI General Embedding) model.
# Uses the bge-large-en model to generate high-quality dense vector embeddings
# optimized for semantic search, retrieval, and RAG applications.
bge_embeddings = HuggingFaceBgeEmbeddings(
    model_name="BAAI/bge-large-en"
)

In [ ]:
# OpenAI embeddings are available if you prefer them over local HuggingFace models
# Uncomment these lines to use text-embedding-ada-002 via the OpenAI API instead
# from langchain_community.embeddings import OpenAIEmbeddings
# openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

### Creating Separate ChromaDB Collections for Each Book

ChromaDB is a local vector database that stores both the raw chunk text and its embedding vector together on disk. We create two completely separate collections inside the same ChromaDB instance, one for Harry Potter and one for GOT.

Keeping them in separate collections is essential for MergerRetriever to work correctly. Each collection will be backed by its own retriever, and the merger will query both independently. If we mixed both books into one collection we would lose the ability to tune or inspect each source separately.

Chroma.from_documents does three things internally. It calls bge_embeddings.embed_documents on every chunk to produce a dense vector. It stores each chunk text and vector together in the collection. It builds a searchable HNSW index (Hierarchical Navigable Small World) over all vectors so future nearest-neighbor queries run in milliseconds rather than scanning every vector linearly.

collection_metadata with hnsw set to cosine tells ChromaDB to use cosine similarity as the distance metric for this index. Cosine similarity is better than Euclidean distance for text because it is not affected by vector magnitude, only by the angle between vectors.

# Ingest the Data into the Chroma Database

In [ ]:
from langchain_community.vectorstores import Chroma
import chromadb

# Get the current working directory as the base for all paths
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)

In [ ]:
# Create a db subfolder inside the current directory to store all ChromaDB data
# All vector index files will be written here and persist between sessions
DB_DIR = os.path.join(CURRENT_DIR, "db")
print(DB_DIR)

In [ ]:
# Configure ChromaDB for persistent local storage
# is_persistent=True: the index is saved to disk and survives kernel restarts
#   without this, all embeddings would need to be recomputed on every run
# persist_directory: the folder where ChromaDB writes its index files
# anonymized_telemetry=False: disables sending usage statistics to Chroma cloud servers
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [ ]:
# Ingest Harry Potter chunks into ChromaDB
# from_documents embeds every chunk using bge_embeddings and stores text plus vector together
# collection_name='harrypotter' namespaces this data separately from the GOT collection
# collection_metadata={'hnsw':'cosine'} sets cosine similarity as the vector distance metric
harrypotter_vectorstore = Chroma.from_documents(
    documents=text_harrypotter,
    embedding=bge_embeddings,
    client_settings=client_settings,
    collection_name="harrypotter",
    collection_metadata={"hnsw": "cosine"},
    persist_directory=os.path.join(DB_DIR, "harrypotter"),
)

In [ ]:
# Ingest GOT chunks into a completely separate ChromaDB collection
# Same settings as Harry Potter but with a different collection_name and persist subdirectory
got_vectorstore = Chroma.from_documents(
    documents=text_got,
    embedding=bge_embeddings,
    client_settings=client_settings,
    collection_name="got",
    collection_metadata={"hnsw": "cosine"},
    persist_directory=os.path.join(DB_DIR, "got"),
)

## Category 4  Individual Retrievers and MergerRetriever (LOTR)

### Wrapping Each Vector Store as a Retriever

as_retriever wraps a ChromaDB vector store behind the standard LangChain retriever interface. This abstraction means the vector store can be swapped out for any other retriever type without changing any downstream code.

search_type='mmr' enables Maximum Marginal Relevance search. Standard similarity search returns the top k chunks closest to the query vector which often produces near-duplicate results since similar sections of a book share similar language. MMR prevents this by iteratively selecting chunks that are both relevant to the query and different from what has already been selected. Each new selection maximises relevance while minimising redundancy with already-chosen chunks, producing a more diverse and informative result set.

k=5 means each individual retriever returns 5 chunks per query. Since we have two retrievers the merged result contains up to 10 chunks before the compression pipeline filters them.

# Create a Retriever for Each Book

In [ ]:
# Create a retriever for the Harry Potter vector store
# search_type='mmr': Maximum Marginal Relevance selects diverse and relevant results
# k=5: return the top 5 most relevant and diverse chunks per query
retriever_harrypotter = harrypotter_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "include_metadata": True}
)

In [ ]:
# Test the Harry Potter retriever in isolation before merging
# invoke sends the query to ChromaDB which embeds it and runs MMR search
result1 = retriever_harrypotter.invoke("Who is Harry Potter?")
result1

In [ ]:
# Display each retrieved chunk with clear separators so we can read all five results
for i, doc in enumerate(result1, start=1):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"{'='*60}")
    print(doc.page_content)

In [ ]:
# Create a separate retriever for the GOT vector store using identical settings
retriever_got = got_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "include_metadata": True}
)

In [ ]:
# Test the GOT retriever in isolation with a GOT-specific character name
result2 = retriever_got.invoke("Who is Ser Waymar ?")
result2

### Merging Both Retrievers With MergerRetriever

MergerRetriever is also called LOTR (Lord of the Retrievers). It accepts a list of any number of individual retrievers and when queried it fans out the same query to all of them simultaneously. The results from every retriever are concatenated into a single flat list of Document objects.

This is the key architectural component that makes multi-source RAG possible without running separate queries manually. The outside world just calls lotr.invoke(query) and gets back combined results from every underlying source in one call.

The raw merged output is intentionally unfiltered at this stage. It contains up to 10 chunks regardless of which book is actually relevant to the question. A question about Harry Potter still returns 5 GOT chunks and vice versa. The next category addresses this by adding a compression pipeline.

# Merge Both Retrievers (Also Called Lord of the Retrievers LOTR)

In [ ]:
from langchain_classic.retrievers import MergerRetriever

# MergerRetriever queries all listed retrievers simultaneously and concatenates their results
# With two retrievers each returning k=5, every query produces up to 10 raw chunks
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

In [ ]:
# Test the merged retriever with a Harry Potter question
# Notice results from both books appear even though only one is relevant
for chunk in lotr.invoke("Who is Harry Potter?"):
    print(chunk.page_content)

In [ ]:
# Test with a GOT-specific question
# Again results from both books are mixed in the raw output
for chunk in lotr.invoke("Who is Ser Waymar?"):
    print(chunk.page_content)

## The Raw Merged Output Is Noisy

The output above reveals the problem with naive merging. When a Harry Potter question is asked the results contain GOT chunks, and when a GOT question is asked Harry Potter chunks appear. Even within one book the MMR results may contain chunks that overlap in content.

This noise wastes precious LLM context window space on irrelevant content and buries the truly relevant chunks in a sea of less relevant ones, which degrades the quality of the generated answer.

The compression pipeline in the next category solves both problems.

## Category 5  Compression Pipeline With EmbeddingsRedundantFilter and LongContextReorder

### Two Problems and Two Solutions

The raw merged output from LOTR has two structural problems that this pipeline solves.

Problem 1 is redundancy. When both retrievers search the same conceptual space some returned chunks will be near-duplicates of each other. Feeding duplicate context to an LLM wastes tokens and can cause the model to overweight that content in its answer.

EmbeddingsRedundantFilter solves this by computing cosine similarity between every pair of retrieved chunks using the bge_embeddings model. Any chunk that is too similar to another already-selected chunk is discarded. The result is a deduplicated set of chunks where each one contributes genuinely new information.

Problem 2 is the lost-in-the-middle phenomenon. Research has consistently shown that LLMs pay the most attention to content at the very beginning and the very end of their context window. Content buried in the middle is systematically underweighted even when it is more relevant than the surrounding content.

LongContextReorder solves this by reordering the deduplicated chunks so the most relevant ones appear at position 1 and the last position, while the least relevant ones are placed in the middle. This positional arrangement ensures the LLM pays maximum attention to the highest-quality content.

DocumentCompressorPipeline chains these two transformers so they execute sequentially: first deduplicate, then reorder the survivors.

ContextualCompressionRetriever wraps the entire system so the downstream LLM chain just sees one retriever that automatically returns clean, deduplicated, optimally ordered context.

# Build the Compression Pipeline for the LLM

In [ ]:
from langchain_community.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder,
)

from langchain_classic.retrievers.document_compressors import (
    DocumentCompressorPipeline,
)

from langchain_classic.retrievers import (
    ContextualCompressionRetriever,
)

In [ ]:
# Step 1: EmbeddingsRedundantFilter removes semantically duplicate chunks
# It uses bge_embeddings to compare every pair of retrieved chunks
# Chunks that are too similar to one already selected are dropped from the final set
# Using the stronger bge embedding here ensures the similarity judgement is accurate
filter = EmbeddingsRedundantFilter(embeddings=bge_embeddings)

# Step 2: LongContextReorder reorders the surviving deduplicated chunks
# The most relevant chunks are moved to the start and end of the list
# The least relevant chunks are placed in the middle
# This positional arrangement counteracts the lost-in-the-middle attention problem in LLMs
reordering = LongContextReorder()

# Chain both transformers sequentially inside a DocumentCompressorPipeline
# transformers are applied in order: filter first removes duplicates, then reordering optimises positions
pipeline = DocumentCompressorPipeline(
    transformers=[filter, reordering]
)

# Wrap everything into a single ContextualCompressionRetriever
# base_retriever: the MergerRetriever (LOTR) that fetches the initial combined candidates
# base_compressor: the pipeline that deduplicates and reorders those candidates
# The outside world calls compression_retriever_reordered.invoke(query) and receives
# a clean, deduplicated, optimally ordered list of chunks ready for the LLM
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline,
    base_retriever=lotr,
    search_kwargs={"k": 3, "include_metadata": True},
)

In [ ]:
# Test the compression retriever with a Harry Potter question
# Compare this output to the noisy raw lotr output from Category 4
docs1 = compression_retriever_reordered.invoke("Who is harry potter?")

print(docs1)

In [ ]:
# Test with a GOT-specific question
docs = compression_retriever_reordered.invoke("Who is Ser Waymar Royce?")

print(docs)

In [ ]:
# Inspect the metadata of each retrieved chunk
# metadata shows which source file and which page number each chunk came from
for doc in docs:
    print(doc.metadata)

In [ ]:
# Display each chunk with its metadata and content in a structured readable format
for i, doc in enumerate(docs, start=1):
    print(f"Document {i}")
    print("Metadata:", doc.metadata)
    print("Content:", doc.page_content)
    print("=" * 100)

In [ ]:
# Check how many chunks survived the redundancy filter for the GOT question
# Should be fewer than the original 10 raw chunks from LOTR
print(len(docs))

In [ ]:
# Check how many chunks survived for the Harry Potter question
print(len(docs1))

## Category 6  LLM Setup and LCEL RAG Chain

### Loading TinyLlama as the Local LLM

TinyLlama-1.1B-Chat-v1.0 is a compact 1.1 billion parameter decoder-only transformer fine-tuned for instruction following and chat. It is small enough to run on a standard laptop CPU without any quantization or GPU, making it an accessible choice for development and experimentation.

AutoModelForCausalLM is the HuggingFace class for loading decoder-only models like LLaMA, Mistral, and Zephyr. These models generate text one token at a time by predicting the most probable next token given everything that came before in the input sequence.

AutoTokenizer loads the paired tokenizer for the model. The tokenizer converts raw text strings into integer token IDs that the model processes, and converts the model output integer IDs back into readable text.

device_map='auto' tells HuggingFace Accelerate to automatically place model layers on the best available hardware, using GPU if available and falling back to CPU otherwise.

torch_dtype='auto' lets the model load in the most efficient precision format available on the current hardware, typically bfloat16 on GPU and float32 on CPU.

# Load the Text Generation Model (LLM) from HuggingFace

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

In [ ]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Define the model name

In [ ]:
# Load the tokenizer for the specified model
# The tokenizer must always match the model exactly so token IDs correspond correctly
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# Load the model weights from HuggingFace Hub
# device_map='auto': Accelerate places layers on GPU if available, CPU otherwise
# torch_dtype='auto': loads in bfloat16 on GPU or float32 on CPU automatically
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto",
)

In [ ]:
# Create the HuggingFace text generation pipeline
# task='text-generation': the model generates new tokens continuing from the input
# max_new_tokens=512: generate at most 512 new tokens beyond the input prompt
# temperature=0.7: controls randomness. Lower values produce more focused deterministic output.
# do_sample=True: use probabilistic sampling rather than always picking the single top token
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
)

In [ ]:
# Wrap the HuggingFace pipeline as a LangChain LLM object
# HuggingFacePipeline implements the standard LangChain LLM interface
# so this llm object plugs into any LangChain chain without any modification
llm = HuggingFacePipeline(
    pipeline=hf_pipeline
)

### Building the RAG Chain Using LCEL

LCEL (LangChain Expression Language) is the modern way to compose chains using the pipe operator. Instead of calling a factory method like RetrievalQA.from_chain_type, you explicitly connect each step using the vertical bar operator, making the data flow visible and easy to modify.

The chain works as follows when invoked with a question string.

RunnableParallel runs two operations simultaneously. The context key pipes the question through compression_retriever_reordered to fetch relevant chunks, then pipes those chunks through format_docs which joins their text with double newlines into a single context string. The question key passes the original question through RunnablePassthrough which means no transformation, just forward the input unchanged.

The resulting dictionary with context and question keys is piped into the ChatPromptTemplate which fills those values into the template and produces a formatted prompt.

The formatted prompt is piped into the LLM which generates an answer token by token.

The generated text is piped into StrOutputParser which extracts the plain string from the LLM output object.

# LCEL Chain (LangChain Expression Language)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Define the prompt template that structures the LLM input
# The {context} placeholder is filled with retrieved and reordered chunks
# The {question} placeholder is filled with the user question
# Instructing the model to answer using only the provided context prevents hallucination
template = (
    "You are a helpful AI assistant.\n\n"
    "Answer the question using only the provided context.\n\n"
    "Context:\n{context}\n\n"
    "Question:\n{question}\n\n"
    "Answer:"
)

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
def format_docs(docs):
    # Join all retrieved chunk texts into one single string separated by double newlines
    # This produces the context block that gets inserted into the {context} placeholder
    # Double newlines between chunks help the LLM distinguish where one chunk ends and another begins
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
# Build the full LCEL RAG chain by connecting all components with the pipe operator
# Step 1: RunnableParallel fetches context and passes the question through simultaneously
# Step 2: The combined dict fills the prompt template
# Step 3: The filled prompt goes to the LLM for answer generation
# Step 4: StrOutputParser extracts the plain text string from the LLM output
retrievel_chain = (
    {
        "context": compression_retriever_reordered | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Running Queries and Inspecting Results

We now run three queries through the complete pipeline. The first tests a Harry Potter question, the second tests a GOT question, and the third retrieves source documents to show exactly which pages and files contributed to the answer.

Each invoke call automatically triggers the full pipeline: compression retriever fetches and filters chunks from both books, the prompt template formats them, TinyLlama generates an answer, and StrOutputParser returns the clean text.

In [ ]:
# Query 1: Harry Potter question through the complete pipeline
response = retrievel_chain.invoke(
    "Who is Harry Potter?"
)

print(response)

In [ ]:
# Query 2: GOT question
query = "who is jon snow?"
results = retrievel_chain.invoke(query)

In [ ]:
# Check the type of the result
# StrOutputParser returns a plain Python string, not a dict
print(type(results))

In [ ]:
# Print the generated answer for the Jon Snow question
print(results)

In [ ]:
# Query 3: Harry Potter question with source document attribution
query = "who is Harry Potter?"
results1 = retrievel_chain.invoke(query)

In [ ]:
# Retrieve the actual source documents used for this query separately
# This shows exactly which chunks from which pages and files contributed to the answer
source_documents = compression_retriever_reordered.invoke(query)

In [ ]:
# Display each source document with its page number and source file path
# This is source attribution that lets you verify the answer is grounded in real content
for i, doc in enumerate(source_documents, start=1):
    print(f"\n{'='*60}")
    print(f"Document {i}")
    print(f"Page   : {doc.metadata.get('page')}")
    print(f"Source : {doc.metadata.get('source')}")
    print("-"*60)
    print(doc.page_content)

## Conclusion

This notebook demonstrated a complete multi-source RAG pipeline built around two core LangChain components: MergerRetriever and LongContextReorder.

MergerRetriever (LOTR) unified two completely independent knowledge bases into a single retriever interface. It answered questions about Harry Potter and Game of Thrones through one invoke call by fanning out to both ChromaDB collections simultaneously. This pattern scales to any number of sources without changing any downstream code.

The raw merged output was intentionally noisy, containing up to 10 chunks from both books regardless of which was relevant. The DocumentCompressorPipeline with EmbeddingsRedundantFilter resolved this by removing semantically duplicate chunks. LongContextReorder further improved the result by placing the most relevant chunks at the start and end of the context window where LLMs pay the most attention. This directly addresses the lost-in-the-middle problem that degrades answer quality in naive RAG systems.

The LCEL chain provided a transparent and modifiable pipeline where every transformation step (retrieval, filtering, reordering, prompting, generation, parsing) was explicit and independently inspectable.

The two-embedding-model strategy (bge-large-en for ingestion and retrieval, all-MiniLM-L6-v2 for the redundancy filter) demonstrated a deliberate performance-quality trade-off that is standard in production RAG systems.

Overall this architecture is the recommended approach when building RAG systems over multiple heterogeneous data sources because it keeps each source independently tunable while presenting a single clean interface to the generation layer.